# Helena's Busplan — GTFS Daten-Extraktion

Dieses Notebook lädt den VMT-GTFS-Feed für Thüringen und extrahiert
die Fahrplandaten für die gewünschten Buslinien.

**Ausführen:** Alle Zellen von oben nach unten ausführen (Run All).
Die Ergebnisse landen in `../data/`.

**Wann ausführen?**  
SNG wechselt den Fahrplan ca. zweimal im Jahr (Juni/Dezember).
Die App zeigt einen Hinweis, wenn der Fahrplan abgelaufen ist.

In [ ]:
# ── Konfiguration ──────────────────────────────────────────────────────────
VMT_GTFS_URL = "https://www.vmt-thueringen.de/fileadmin/VMT_Redaktion/OPEN_DATA/VMT_GTFS.zip"
SNG_AGENCY_ID = "73"  # Städtische Nahverkehrsgesellschaft mbH Suhl/Zella-Mehlis
LINES = ["D1", "D2"]  # Weitere Linien hier eintragen, z.B. ["D1","D2","A1"]

import csv, json, zipfile, os, io
from urllib.request import urlretrieve
from collections import defaultdict
from pathlib import Path

OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Ausgabeverzeichnis: {OUTPUT_DIR.resolve()}")

In [ ]:
# ── GTFS herunterladen ─────────────────────────────────────────────────────
import tempfile

gtfs_path = Path(tempfile.gettempdir()) / "vmt_gtfs.zip"
print(f"Lade GTFS von {VMT_GTFS_URL} ...")
urlretrieve(VMT_GTFS_URL, gtfs_path)
size_mb = gtfs_path.stat().st_size / 1_048_576
print(f"Heruntergeladen: {size_mb:.1f} MB → {gtfs_path}")

In [ ]:
# ── CSV-Dateien einlesen ───────────────────────────────────────────────────
def read_csv(zf, name):
    with zf.open(name) as f:
        content = f.read().decode("utf-8-sig")
        return list(csv.DictReader(content.splitlines()))

def normalize_time(t):
    if not t: return None
    parts = t.strip().split(":")
    h, m = int(parts[0]) % 24, int(parts[1])
    return f"{h:02d}:{m:02d}"

def get_day_type(row):
    if row.get("saturday") == "1": return "saturday"
    if row.get("sunday")   == "1": return "sunday"
    for day in ["monday","tuesday","wednesday","thursday","friday"]:
        if row.get(day) == "1": return "weekday"
    return None

with zipfile.ZipFile(gtfs_path) as zf:
    routes    = read_csv(zf, "routes.txt")
    trips     = read_csv(zf, "trips.txt")
    stops     = read_csv(zf, "stops.txt")
    st        = read_csv(zf, "stop_times.txt")
    calendar  = read_csv(zf, "calendar.txt")
    try:
        feed_info = read_csv(zf, "feed_info.txt")
    except KeyError:
        feed_info = [{}]

def fmt_date(d): return f"{d[:4]}-{d[4:6]}-{d[6:8]}" if d else ""
valid_from  = fmt_date(feed_info[0].get("feed_start_date", ""))
valid_until = fmt_date(feed_info[0].get("feed_end_date", ""))
print(f"Fahrplan gültig: {valid_from} bis {valid_until}")

service_day = {r["service_id"]: get_day_type(r) for r in calendar}
stop_names  = {s["stop_id"]: s["stop_name"] for s in stops}

target_routes = {r["route_short_name"]: r["route_id"]
                 for r in routes
                 if r.get("agency_id") == SNG_AGENCY_ID
                 and r.get("route_short_name") in LINES}
print(f"Gefundene Linien: {target_routes}")

In [ ]:
# ── Für jede Linie JSON erzeugen ───────────────────────────────────────────
for line_name in LINES:
    route_id = target_routes.get(line_name)
    if not route_id:
        print(f"WARNUNG: Linie {line_name} nicht gefunden!")
        continue

    print(f"\n=== Linie {line_name} (route_id={route_id}) ===")
    line_trips = [t for t in trips if t["route_id"] == route_id]
    all_trip_ids = {t["trip_id"] for t in line_trips}

    trip_headsigns = {t["trip_id"]: t.get("trip_headsign", "") for t in line_trips}

    # stop_times für diese Linie
    trip_st = defaultdict(list)
    for row in st:
        if row["trip_id"] in all_trip_ids:
            trip_st[row["trip_id"]].append((
                int(row["stop_sequence"]),
                row["stop_id"],
                row.get("departure_time") or row.get("arrival_time", "")
            ))
    for tid in trip_st:
        trip_st[tid].sort(key=lambda x: x[0])

    # Trips nach Richtung und Tagestyp gruppieren
    dir_trips = defaultdict(lambda: defaultdict(list))
    for t in line_trips:
        did = t.get("direction_id", "0")
        dt  = service_day.get(t["service_id"])
        if dt:
            dir_trips[did][dt].append(t["trip_id"])

    directions_out = []
    for did in sorted(dir_trips.keys()):
        all_dir_trips = [t for ts in dir_trips[did].values() for t in ts]
        if not all_dir_trips: continue

        # Kanonische Haltestellenreihenfolge = längste Fahrt
        canonical = max(all_dir_trips, key=lambda t: len(trip_st[t]))
        stop_ids_ord  = [s[1] for s in trip_st[canonical]]
        stop_names_ord = [stop_names.get(sid, sid) for sid in stop_ids_ord]

        # Häufigstes Zielschild
        hs_list = [trip_headsigns.get(t,"") for t in all_dir_trips if trip_headsigns.get(t)]
        headsign = max(set(hs_list), key=hs_list.count) if hs_list else f"Richtung {did}"

        print(f"  Richtung {did} ({headsign}): {len(stop_names_ord)} Haltestellen")
        print(f"  Strecke: {stop_names_ord[0]} -> {stop_names_ord[-1]}")

        schedules = {}
        for dt, tid_list in dir_trips[did].items():
            matrix = []
            for tid in tid_list:
                st_map = {s[1]: normalize_time(s[2]) for s in trip_st[tid]}
                row = [st_map.get(sid) for sid in stop_ids_ord]
                if row[0]: matrix.append(row)
            matrix.sort(key=lambda r: r[0] or "99:99")
            schedules[dt] = matrix
            print(f"    {dt}: {len(matrix)} Fahrten")

        directions_out.append({
            "id": f"dir{did}",
            "direction_id": did,
            "headsign": headsign,
            "stops": stop_names_ord,
            "schedules": schedules
        })

    out = {
        "line": line_name,
        "valid_from": valid_from,
        "valid_until": valid_until,
        "directions": directions_out
    }
    out_path = OUTPUT_DIR / f"{line_name.lower()}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(f"  Gespeichert: {out_path} ({out_path.stat().st_size/1024:.1f} KB)")

print("\nFertig! Bitte 2-3 Zeiten manuell gegen das PDF prüfen.")

## Manuelle Verifikation

Vergleiche ein paar Abfahrtszeiten mit den offiziellen PDFs:
- D1: https://www.sngonline.de/static/sngbf/sng-includes/fahrplan-pdf/D1.pdf
- D2: https://www.sngonline.de/static/sngbf/sng-includes/fahrplan-pdf/D2.pdf

Haltestelle **"Goldlauter, Suhler Straße"** prüfen:
- D1: Index 10 im stops-Array
- D2: Index 18 im stops-Array

In [ ]:
# ── Schnell-Check: Abfahrten von Goldlauter, Suhler Straße ─────────────────
TARGET_STOP = "Goldlauter, Suhler Straße"

for line_name in LINES:
    out_path = OUTPUT_DIR / f"{line_name.lower()}.json"
    if not out_path.exists(): continue
    with open(out_path, encoding="utf-8") as f:
        data = json.load(f)
    print(f"\n=== Linie {line_name} — Abfahrten von '{TARGET_STOP}' ===")
    for d in data["directions"]:
        if TARGET_STOP not in d["stops"]: continue
        idx = d["stops"].index(TARGET_STOP)
        print(f"  Richtung: {d['headsign']} (Stop-Index {idx})")
        for dt, trips_matrix in d["schedules"].items():
            times = [row[idx] for row in trips_matrix if row[idx]]
            print(f"  {dt:10s}: {', '.join(times)}")